# Optimal Sampling Strateges - Constant birth term

In this notebook we use synthetic viral read count data from a fully-parameterised toy population to theoretically assess determine optimal sampling study strategies, when the rodent population are assumed to follow the dynamics of the SIR algorithm with contant birth term rate. If the estimates of the population model parameters are close to the true model parameter values that produced the toy population in the first place imply the validity of inferential approach, and therefore lend credibility when the same pipeline is used with real metaviromic data, as done in _James Hay et al. (2021)[1]_.

Similar to field studies, random samples of rodents are drawn from the simulated toy population at predifined sampling times, which satisfy the following:
 - same total number of rodents sampled at each time point;
 - the sampled individuals can be either susceptible (S), infected (I) or recovered (R), with no predefined quantities of each;
 - all individuals sampled are born and alive at the time of sampling.

For each of the sampled individuals, we use the SIR model's embedded `viral_read_model` to produce viral read count data, similar to what data is produced from the field studies (byproduct in our analyses, ground truth in real studies).

Two parameter inference approaches are evaluated:
 - (1) an optimisation approach, using the CMA-ES method from *Pints [2]*, and 
 - (2) a sampling approach, using the HaarioBardenetACMC method from *Pints [2]*.

We replicate these analyses for a range of sample sizes and frequencies of sampling values, to compare the quality of parameter estimation and proportion of infected population across different sampling protocols.

**************
### References
[1] James A. Hay et al., _Estimating epidemiologic dynamics from cross-sectional viral load distributions_. Science373,**eabh0635(2021)**. DOI:10.1126/science.abh0635

[2] Clerx, M., Robinson, M., Lambert, B., Lei, C. L., Ghosh, S., Mirams, G. R., & Gavaghan, D. J.,
_Probabilistic Inference on Noisy Time Series (PINTS)_.
Journal of Open Research Software (2019), 7(1), 23. DOI:10.5334/jors.252

In [1]:
# Load necessary libraries
import numpy as np
import pandas as pd
from scipy.stats import multinomial, skew, gumbel_r
import math
import metavirommodel as mm
import metavirommodel.inference as mmi
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pints
from matplotlib import pyplot as plt
import pints.plot

# Choose array of colours for graphs and compartments names
colours = ['blue', 'red', 'green', 'purple', 'orange', 'black', 'gray', 'pink']
compartments = ['S', 'I', 'R']

# Set random seed
np.random.seed(9)

## Gillespie stochastic SIR algorithm with contant birth term rate

#### Define rodent population

In [2]:
# Set initial reproduction number
R_0 = 3

# Set initial population state S - I - R
N_init = 400
# S_init = int(N_init / R_0)
S_init = 380
I_init = N_init - S_init
R_init = 0
initial_population = [S_init, I_init, R_init]

# Set birth rate
theta = 0.05

# Set death rates
mu = 0.002
nu = 0

# Set transition rates
infect_period = 15
beta =  R_0 / infect_period
gamma = 1 / infect_period

# Coalesce into paramater vector
parameters = initial_population
parameters.extend([theta, mu, nu, beta, gamma])

# Instantiate algorithm
algorithm = mm.Metaviromodel()

# Select start and end times
start_time = 1
end_time = 360

times = list(range(start_time, end_time+1))

# Select number of experiments
num_experiments = 1

output_algorithm = []

S_history_algorithm = []
I_history_algorithm = []
R_history_algorithm = []

I_times_history_algorithm = []
R_times_history_algorithm = []

for _ in range(num_experiments):
    output, S_history, I_history, R_history, I_times_history, R_times_history = algorithm.simulate_fixed_times(parameters, start_time, end_time)
    output_algorithm.append(output)

    S_history_algorithm.append(S_history)
    I_history_algorithm.append(I_history)
    R_history_algorithm.append(R_history)

    I_times_history_algorithm.append(I_times_history)
    R_times_history_algorithm.append(R_times_history)

output_algorithm = np.asarray(output_algorithm)

### Plot output of Gillespie for the different compartments

In [3]:
# Trace names - represent the type of individuals for the simulation
trace_name = ['{}'.format(s) for s in compartments]

# Names of panels
panels = ['{} only'.format(s) for s in compartments] + ['Total Population']

fig = go.Figure()
fig = make_subplots(rows=int(np.ceil(len(panels)/2)), cols=2, subplot_titles=tuple('{}'.format(p) for p in panels))

# Add traces to the separate counts panels
for s, spec in enumerate(compartments):
    fig.add_trace(
        go.Scatter(
            y=np.mean(output_algorithm[:, :, s], axis=0).tolist(),
            x=times,
            mode='lines',
            name=trace_name[s],
            line_color=colours[s]
        ),
        row= int(np.floor(s / 2)) + 1,
        col= s % 2 + 1
    )

fig.add_trace(
    go.Scatter(
        y=np.mean(np.sum(output_algorithm, axis=2), axis=0).tolist(),
        x=times,
        mode='lines',
        name='Total Population',
        line_color='black'
    ),
    row= 2,
    col= 2
)

# Add axis labels
fig.update_layout(
    title='Counts of compartments over time:<br>IC = {}, θ = {}, μ = {}, v = {}, β = {:.2f}, γ = {:.2f}'.format(parameters[0:3], parameters[3], parameters[4], parameters[5], parameters[6], parameters[7]),
    width=1100, 
    height=600,
    plot_bgcolor='white',
    xaxis=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis=dict(
        linecolor='black',
        title = 'Individuals'),
    xaxis2=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis2=dict(
        linecolor='black',
        title = 'Individuals'),
    xaxis3=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis3=dict(
        linecolor='black',
        title = 'Individuals'),
    xaxis4=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis4=dict(
        linecolor='black',
        title = 'Individuals')
    #legend=dict(
    #    orientation="h",
    #    yanchor="bottom",
    #    y=1.02,
    #    xanchor="right",
    #    x=1
    #)
    )

fig.write_image('images/Optimal-sampling-gillespie.pdf')
fig.show()

## Produce Viral read counts values

In [4]:
# Set parameter for the viral read counts model
t_eclipse = 3  # (0 days) Time from infection to initial viral growth
t_peak = 7  # (5 days ) Time from initial viral growth to peak viral load
t_switch = 5  # (9.38 days) Time from peak viral load to secondary waning phase
t_mod = 15  # (14 days) time from secondary waning phase until gumbel distribution reaches its min scale parameter
t_LOD = math.inf # ( inf days ) Time from infection until modal read counts value is equal to the limit of detection

sigma_obs = 0.25  # Initial scale parameter for the Gumbel distribution until a=teclipse+tpeak+tswitch
s_mod = 0.4  # 0.4 multiplicative factor applied to scale paramter for the Gumble distrbution - starting at t_eclipse + t_peak + t_switch + t_scle
v_zero = 2  # read counts value at time of infection
v_peak = 3880  # (20) Modal read counts value at peak viral load
v_switch = 480  # (33) Modal read counts value at a = teclipse + tpeak + tswitch
v_LOD = 2  # Limit of detection of read counts value

parameters_vl = [
    t_eclipse, t_peak, t_switch, t_mod, t_LOD,
    v_zero, v_peak, v_switch, v_LOD,
    s_mod, sigma_obs]

# Set read counts value for the suceptible and recovered individuals
VR_susc = 0

### Plot Viral read Model

In [5]:
time_from_infec = np.arange(1, 50)
vr_val = []

for ti in time_from_infec:
    ti_vr_val = []
    for _ in range(10000):
        ti_vr_val.append(algorithm.viral_read_model(parameters_vl, ti))
    vr_val.append(ti_vr_val)

vr_val = np.asarray(vr_val)

In [6]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y=time_from_infec,
        x=np.mean(vr_val, axis=1),
        mode='lines',
        name='Mean Viral read',
        showlegend=False,
    )
)

fig.add_trace(
    go.Scatter(
        y=time_from_infec.tolist() + time_from_infec.tolist()[::-1],
        x=np.quantile(vr_val, 0.975, axis=1).tolist() + np.quantile(vr_val, 0.025, axis=1).tolist()[::-1],
        mode='lines',
        fill='toself',
        fillcolor='blue',
        line_color='blue',
        opacity=0.3,
        showlegend=False,
    )
)

# Add axis labels
fig.update_layout(
    width=500, 
    height=500,
    plot_bgcolor='white',
    xaxis=dict(
        linecolor='black',
        title = 'Mean Viral read',
        autorange='reversed'
        ),
    yaxis=dict(
        linecolor='black',
        title = 'Time since infection'),
    #legend=dict(
    #    orientation="h",
    #    yanchor="bottom",
    #    y=1.02,
    #    xanchor="right",
    #    x=1
    #)
    )

fig.write_image('images/Optimal_sampling_Viral_read_model.pdf')
fig.show()

### Compute the history of recovered individuals that fully clear the virus and generation times distribution

#### 0 = 'not cleared'; 1 = 'cleared'

In [7]:
# Daily probability of recovered fully clearing the virus
p_addl = 0.2

R_history_clear_algorithm = []

# Go through each run experiment
for _ in range(num_experiments):
    R_history_clear = []

    # Go through each recorded day
    for t, time in enumerate(times):
        current_clear_status = []

        # If there are any recovered individual
        if len(R_times_history_algorithm[_][t]) > 0:
            # Go through each of them and
            for ind, ind_ID in enumerate(R_history_algorithm[_][t]):
                clear_status = 0

                # If they have previously cleared the virus they signal that
                if ind_ID in R_history_algorithm[_][t-1] and R_history_clear[-1][R_history_algorithm[_][t-1].index(ind_ID)] == 1:
                    clear_status = 1
                # if not, they could do it today, if their time since infection exceeds teclipse + tpeak + tswitch
                elif time > R_times_history_algorithm[_][t][ind] + t_eclipse + t_peak + t_switch:
                    clear_status = 1 - np.random.binomial(1, p = (1-p_addl)**(
                        time - R_times_history_algorithm[_][t][ind] - t_eclipse - t_peak - t_switch))

                current_clear_status.append(clear_status)

        R_history_clear.append(current_clear_status)                

    R_history_clear_algorithm.append(R_history_clear)

In [8]:
# Compute the generation times distribution, which also follows a
# right-skewed Gumbel distribution
generation_times = []

for _ in range(70):
    if _ < t_eclipse + t_peak + t_switch:
        generation_times.append(
            1-gumbel_r.cdf(
                np.log(v_LOD),
                algorithm._compute_mode_vr_model(
                    _, t_eclipse, t_peak, t_switch, t_LOD,
                    np.log(v_zero), np.log(v_peak), np.log(v_switch), np.log(v_LOD)),
                algorithm._compute_sigma_vr_model(
                    _, t_eclipse, t_peak, t_switch, t_mod,
                    s_mod, sigma_obs)
            ))
        
    else:
        generation_times.append(
            (1-gumbel_r.cdf(
                np.log(v_LOD),
                algorithm._compute_mode_vr_model(
                    _, t_eclipse, t_peak, t_switch, t_LOD,
                    np.log(v_zero), np.log(v_peak), np.log(v_switch), np.log(v_LOD)),
                algorithm._compute_sigma_vr_model(
                    _, t_eclipse, t_peak, t_switch, t_mod,
                    s_mod, sigma_obs)
            )) * (1-p_addl)**(_ - t_eclipse - t_peak - t_switch))

#### Plot generation times

In [9]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=time_from_infec,
        y=generation_times,
        mode='lines',
        name='Generation times',
        showlegend=False,
    )
)

fig.show()

## Parameter inference
In this section we test the quality of parameter inference for two distinct inference approaches: 

- (1) an optimisation approach, using the CMA-ES method from *Pints [2]*, and 
 - (2) a sampling approach, using the HaarioBardenetACMC method from *Pints [2]*,

for a range of sample sizes and frequencies of sampling values, to compare the quality of parameter estimation and proportion of infected population across different sampling protocols.

#### Sample individuals with specific frequencies and in specific batch sizes

In [10]:
freq_samplying_range = [14, 28, 42, 56]
sample_size_range = [15, 20, 25]

### 1. Optimisation method

#### Method to create Ct value data and ground truth

In [11]:
def sensitivity_analysis_run(sample_points, sample_size):
    vr_values = []
    vr_infec = []

    vr_susc_ids = []
    vr_infec_ids = []
    vr_recov_ids = []

    vr_time_of_recov_infec = []
    vr_time_of_infec = []
    vr_time_since_infec = []

    for _ in range(num_experiments):
        experiment_vr_values = []
        experiment_infec = []

        experiment_susc_ids = []
        experiment_infec_ids = []
        experiment_recov_ids = []

        experiment_time_of_recov_infec = []
        experiment_time_of_infec = []
        experiment_time_since_infec = []
        # At each point in time sample sample_size individuals
        for time in sample_points:
            # Identify the current infections at the specified timepoint
            current_susceptibles = S_history_algorithm[_][time-1]
            current_infections = I_history_algorithm[_][time-1]
            current_recovered = R_history_algorithm[_][time-1]
            current_infection_times = I_times_history_algorithm[_][time-1]
            current_recov_infection_times = R_times_history_algorithm[_][time-1]
            current_recov_clear_virus_status = R_history_clear_algorithm[_][time-1]
            current_recov_clear_virus_status = R_history_clear_algorithm[_][time-1]

            # Sample without replacement the sample_size individuals and
            # determine their time since infection to produce Ct values
            number_selected_susc, number_selected_infec, number_selected_rec = \
                multinomial.rvs(
                    n=sample_size,
                    p=output_algorithm[_, time-1, :]/np.sum(output_algorithm[_, time-1, :])) # determine how many of those sampled are S, I and R

            # First add the Ct values for the sampled susceptibele and recovered individuals
            sampled_vr_values = [VR_susc] * number_selected_susc

            selected_individuals_susc_ids = np.random.choice(
                    current_susceptibles,
                    size=number_selected_susc,
                    replace=False).tolist() # determine the ids of those sampled Ss
            
            if len(current_recov_infection_times) > 0:
                # If we have at least one selected recovered
                selected_individuals_indices = np.random.choice(
                    range(len(current_recov_infection_times)),
                    size=number_selected_rec,
                    replace=False).tolist() # determine the indices of those sampled Rs
            
                selected_individuals_rec_ids = [current_recovered[_] for _ in selected_individuals_indices]

                # Determine the time of infection of those sampled Rs
                selected_individuals_recov_infec_times = [current_recov_infection_times[_] for _ in selected_individuals_indices]

                sample_time_since_infec = time - selected_individuals_recov_infec_times # determine how long since infection for selected Rs

                # Determine the clearence of infection of those sampled Rs
                selected_individuals_clear_virus_status = [current_recov_clear_virus_status[_] for _ in selected_individuals_indices]

                # Run viral read model to determine individual viral read counts for each sample
                for i, ti in enumerate(sample_time_since_infec):
                    sampled_vr_values.append(algorithm.viral_read_model(parameters_vl, ti) * selected_individuals_clear_virus_status[i])

            elif number_selected_rec > 0:
                # If initial step when no history of infection is provided
                for i in range(number_selected_rec):
                    sampled_vr_values.append(algorithm.viral_read_model(parameters_vl, time))
                
                sample_time_since_infec = np.zeros(number_selected_rec)
                selected_individuals_rec_ids = [] 
            else:
                sample_time_since_infec = []
                selected_individuals_rec_ids = []

            if len(current_infection_times) > 0:
                # If we have at least one selected infection
                selected_individuals_indices = np.random.choice(
                    range(len(current_infection_times)),
                    size=number_selected_infec,
                    replace=False).tolist() # determine the indices of those sampled Is
                
                # Determine the ids of those sampled Is
                selected_individuals_infec_ids = [current_infections[_] for _ in selected_individuals_indices]

                # Determine the time of infection of those sampled Is
                selected_individuals_infec_times = [current_infection_times[_] for _ in selected_individuals_indices]
            
                sample_time_since_infec = time - selected_individuals_infec_times # determine how long since infection for selected Is

                # Run Ct model to determine individual Ct counts for each sample
                for ti in sample_time_since_infec:
                    sampled_vr_values.append(algorithm.viral_read_model(parameters_vl, ti))
            
            elif number_selected_infec > 0:
                # If initial step when no history of infection is provided
                for i in range(number_selected_infec):
                    sampled_vr_values.append(algorithm.viral_read_model(parameters_vl, time))
                
                selected_individuals_infec_times = np.zeros(number_selected_infec)
                sample_time_since_infec = np.zeros(number_selected_infec)
                selected_individuals_infec_ids = [] 
            else:
                selected_individuals_infec_times = []
                sample_time_since_infec = []
                selected_individuals_infec_ids = [] 

            experiment_vr_values.append(sampled_vr_values)
            experiment_infec.append(number_selected_infec)
            
            experiment_susc_ids.append(selected_individuals_susc_ids)
            experiment_infec_ids.append(selected_individuals_infec_ids)
            experiment_recov_ids.append(selected_individuals_rec_ids)

            experiment_time_of_recov_infec.append(selected_individuals_recov_infec_times)
            experiment_time_of_infec.append(selected_individuals_infec_times)
            experiment_time_since_infec.append(sample_time_since_infec)
        
        vr_values.append(experiment_vr_values)
        vr_infec.append(experiment_infec)

        vr_susc_ids.append(experiment_susc_ids)
        vr_infec_ids.append(experiment_infec_ids)
        vr_recov_ids.append(experiment_recov_ids)

        vr_time_of_recov_infec.append(experiment_time_of_recov_infec)
        vr_time_of_infec.append(experiment_time_of_infec)
        vr_time_since_infec.append(experiment_time_since_infec)

    vr_values = np.asarray(vr_values)
    vr_infec = np.asarray(vr_infec)

    vr_time_of_infec_data = []

    for _ in range(num_experiments):
        experiment_vr_time_of_infec_data = pd.DataFrame(columns=['ID', 'Value'])
        for t, time in enumerate(sample_points):
            experiment_vr_time_of_infec_data = pd.concat(
                [
                    experiment_vr_time_of_infec_data,
                    pd.DataFrame({
                        'ID': vr_susc_ids[_][t] + vr_recov_ids[_][t] + vr_infec_ids[_][t],
                        'Value': [400] * len(vr_susc_ids[_][t]) + vr_time_of_recov_infec[_][t] + vr_time_of_infec[_][t]
                    })
                ])
            
        vr_time_of_infec_data.append(experiment_vr_time_of_infec_data)

    vr_values_data = []

    for _ in range(num_experiments):
        experiment_vr_values_data = pd.DataFrame(columns=['ID', 'TimeOfSample', 'Value'])
        for t, time in enumerate(sample_points):
            experiment_vr_values_data = pd.concat(
                [
                    experiment_vr_values_data,
                    pd.DataFrame({
                        'ID': vr_susc_ids[_][t] + vr_recov_ids[_][t] + vr_infec_ids[_][t],
                        'TimeOfSample': [time] * sample_size,
                        'Value': vr_values[_, t, :].tolist()
                    })
                ])
            
        vr_values_data.append(experiment_vr_values_data)

    mvr_inference = mmi.MVRVirReadInfer(algorithm, generation_times=generation_times)

    # Read Vireal read counts and Ct values data
    mvr_inference.read_viral_read_data(vr_values_data[0], parameters_vl)

    R0_found = mvr_inference.optimisation_problem_setup()[0]

    shody_recov_freq = []

    for t in range(vr_values[0].shape[0]):
        shody_recov_freq.append((np.where((vr_values[0][t, :] > 130) & (vr_values[0][t, :] < 150))[0]).shape[0] /sample_size)

    return vr_values_data, R0_found, shody_recov_freq, vr_infec[0, :] / sample_size

In [12]:
# Transform birth rate and death rate of infected into function format for inference method
parameters[3] = lambda _: theta
parameters[5] = lambda _: nu

#### Method to run inference with Viral read count data and plot inferred trajectories against ground truth

In [13]:
def routine_run(freq_samplying, sample_size):
    sample_points = np.arange(20, 110, freq_samplying)

    # For each choice of sample size and frequency infer parameters: 
    vr_values_data, R0_found, shody_recov_freq, infec_freq_sample = sensitivity_analysis_run(sample_points, sample_size)

    mvr_inference = mmi.MVRVirReadInfer(algorithm, generation_times=generation_times)
    mvr_inference.read_viral_read_data(vr_values_data[0], parameters_vl)
    mvr_inference._create_posterior()

    parameters[4] = R0_found[1]
    parameters[6] = R0_found[0] / infect_period

    theta_found = []
    infec_found = []
    for _ in range(1000):
        output_found = algorithm.simulate_fixed_times(parameters, start_time, end_time)[0]

        theta_found.append(np.divide(output_found[:, 1], np.sum(output_found, axis=1)))
        infec_found.append(output_found[:, 1])

    theta_found = np.array(theta_found)
    infec_found = np.array(infec_found)

    theta_found_mean = np.mean(theta_found, axis=0)
    theta_found_upper = np.quantile(theta_found, 0.975, axis=0)
    theta_found_lower = np.quantile(theta_found, 0.025, axis=0)

    infec_found_mean = np.mean(infec_found, axis=0)
    infec_found_upper = np.quantile(infec_found, 0.975, axis=0)
    infec_found_lower = np.quantile(infec_found, 0.025, axis=0)

    output_found_det = mvr_inference.loglikelihood._run_sir_model(
        parameters, np.arange(max(mvr_inference.loglikelihood._vr_sampled_times))
    )
    theta_found_det = np.divide(output_found_det[:, 1], np.sum(output_found_det, axis=1))
    infec_found_det = output_found_det[:, 1]

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            y=(output_algorithm[0, :, 1] / np.mean(np.sum(output_algorithm, axis=2), axis=0)).tolist(),
            x=times,
            mode='lines',
            name='True freq',
            line_color='red'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=infec_freq_sample.tolist(),
            x=sample_points,
            mode='lines',
            name='Sample freq',
            line_color='blue'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=(1 - np.array(shody_recov_freq)).tolist(),
            x=sample_points,
            mode='lines',
            name='Sample freq (extreme I)',
            line_color='green'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=theta_found_det.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq - det',
            line_color='black',
            line_dash='dash'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=theta_found_mean.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq',
            line_color='black'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=times + times[::-1],
            y=theta_found_upper.tolist() + theta_found_lower.tolist()[::-1],
            fill='toself',
            fillcolor='black',
            line_color='black',
            opacity=0.3,
            mode='lines',
            showlegend=False,
            name='Inferred freq'
        )
    )

    # Add axis labels
    fig.update_layout(
        title='Sensitivity analysis: Frequency:{} days; Sample size:{}'.format(freq_samplying, sample_size),
        width=700, 
        height=400,
        plot_bgcolor='white',
        xaxis=dict(
            linecolor='black',
            title = 'Time (days)'
            ),
        yaxis=dict(
            linecolor='black',
            title = 'Individuals'),
        )

    fig.write_image('images/Optimal_sampling_Viral_read_CredInt_Freq_{}_Sample_size_{}.pdf'.format(freq_samplying, sample_size))
    fig.show()

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            y=(output_algorithm[0, :, 1]).tolist(),
            x=times,
            mode='lines',
            name='True freq',
            line_color='red'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=infec_found_det.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq - det',
            line_color='black',
            line_dash='dash'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=infec_found_mean.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq',
            line_color='black'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=times + times[::-1],
            y=infec_found_upper.tolist() + infec_found_lower.tolist()[::-1],
            fill='toself',
            fillcolor='black',
            line_color='black',
            opacity=0.3,
            mode='lines',
            showlegend=False,
            name='Inferred freq'
        )
    )

    # Add axis labels
    fig.update_layout(
        title='Total Infections: Frequency:{} days; Sample size:{}'.format(freq_samplying, sample_size),
        width=700, 
        height=400,
        plot_bgcolor='white',
        xaxis=dict(
            linecolor='black',
            title = 'Time (days)'
            ),
        yaxis=dict(
            linecolor='black',
            title = 'Individuals'))

    fig.write_image('images/Total_Infec_SIR_Viral_read__Freq_{}_Sample_size_{}.pdf'.format(freq_samplying, sample_size))
    fig.show()

#### Run optimisation-based inference method for multiple sampling protcols

In [14]:
routine_run(freq_samplying_range[0], sample_size_range[0])

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:144: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:160: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6
Iter. Eval. Best      Current   Time    
0     6     -303.1351 -303.1351   0:02.6
1     12    -303.1351 -304.62     0:04.3
2     18    -303.1351 -303.7775   0:07.0
3     24    -303.1351 -303.2277   0:09.1
20    126   -302.7639 -302.7963   0:38.2
40    246   -244.9496 -245.2095   1:11.2
60    366   -243.3871 -243.4423   1:46.9
80    486   -243.3803 -243.3824   2:20.6
100   606   -243.379  -243.379    2:53.9
120   726   -243.379  -243.379    3:28.9
140   846   -243.379  -243.379    4:00.5
151   906   -243.379  -243.379    4:18.1
Halting: No significant change in best function evaluation for 100 iterations.
[1.87597753e+00 1.00000001e-03] -243.37895627279056
Optimisation phase is finished.


In [15]:
routine_run(freq_samplying_range[1], sample_size_range[0])

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:144: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:160: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6
Iter. Eval. Best      Current   Time    
0     6     -175.444  -175.444    0:01.4
1     12    -175.321  -175.321    0:02.8
2     18    -175.077  -175.077    0:04.2
3     24    -174.3657 -174.3657   0:05.4
20    126   -174.2157 -174.2427   0:20.3
40    246   -170.897  -170.897    0:38.5
60    366   -134.9969 -135.0653   0:57.6
80    486   -134.508  -134.5111   1:17.2
100   606   -134.506  -134.506    1:36.6
120   726   -134.506  -134.506    1:54.2
140   846   -134.506  -134.506    2:11.9
155   930   -134.506  -134.506    2:25.1
Halting: No significant change in best function evaluation for 100 iterations.
[1.79006583e+00 1.00000034e-03] -134.50598721086953
Optimisation phase is finished.


In [16]:
routine_run(freq_samplying_range[2], sample_size_range[0])

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:144: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:160: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6
Iter. Eval. Best      Current   Time    
0     6     -139.4982 -139.4982   0:01.0
1     12    -139.1713 -139.1713   0:01.9
2     18    -139.1713 -139.6912   0:02.6
3     24    -139.1713 -139.7629   0:03.1
20    126   -139.0639 -139.0639   0:13.6
40    246   -137.2578 -137.2578   0:27.1
60    366   -102.4671 -102.4771   0:41.9
80    486   -102.4671 -102.4671   0:58.8
100   606   -102.4671 -102.4671   1:15.6
120   726   -102.4671 -102.4671   1:32.6
140   846   -102.4671 -102.4671   1:49.2
151   906   -102.4671 -102.4671   1:58.3
Halting: No significant change in best function evaluation for 100 iterations.
[1.65891298 0.00268831] -102.4671043154492
Optimisation phase is finished.


In [17]:
routine_run(freq_samplying_range[3], sample_size_range[0])

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:144: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:160: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6
Iter. Eval. Best      Current   Time    
0     6     -64.07837 -64.07837   0:00.5
1     12    -63.79167 -63.79167   0:00.9
2     18    -63.79167 -64.04624   0:01.2
3     24    -63.79167 -64.13695   0:01.6
20    126   -63.79167 -63.80704   0:08.1
40    246   -63.77966 -63.77966   0:15.4
60    366   -49.44444 -49.47293   0:23.8
80    486   -49.16595 -49.16758   0:32.9
100   606   -49.16233 -49.16241   0:40.5
120   726   -49.16225 -49.16226   0:48.3
140   846   -49.16225 -49.16225   0:56.2
158   948   -49.16225 -49.16225   1:02.9
Halting: No significant change in best function evaluation for 100 iterations.
[1.88711460e+00 1.00000001e-03] -49.162253036274386
Optimisation phase is finished.


In [18]:
routine_run(freq_samplying_range[0], sample_size_range[1])

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:144: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:160: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6
Iter. Eval. Best      Current   Time    
0     6     -386.248  -386.248    0:03.4
1     12    -385.1158 -385.1158   0:06.2
2     18    -385.1158 -385.4064   0:08.4
3     24    -385.1158 -385.6284   0:10.9
20    126   -384.7703 -384.7852   0:45.1
40    246   -384.7703 -384.778    1:28.5
60    366   -384.6322 -384.6322   2:13.7
80    486   -306.2342 -306.833    2:58.1
100   606   -306.2224 -306.2224   3:52.5
120   726   -306.1655 -306.1655   4:36.9
140   846   -306.1651 -306.1651   5:17.4
160   966   -306.1651 -306.1651   5:57.8
176   1056  -306.1651 -306.1651   6:31.7
Halting: No significant change in best function evaluation for 100 iterations.
[1.84639495e+00 1.00000003e-03] -306.1650996372371
Optimisation phase is finished.


In [19]:
routine_run(freq_samplying_range[1], sample_size_range[1])

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:144: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:160: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6
Iter. Eval. Best      Current   Time    
0     6     -232.0996 -232.0996   0:01.9
1     12    -232.0996 -232.3464   0:03.5
2     18    -232.0996 -232.1147   0:05.1
3     24    -231.9373 -231.9373   0:06.6
20    126   -231.6068 -231.6382   0:27.4
40    246   -188.4516 -188.4516   0:53.3
60    366   -180.1411 -180.1411   1:18.4
80    486   -180.1317 -180.1323   1:45.8
100   606   -180.1316 -180.1316   2:10.9
120   726   -180.1316 -180.1316   2:35.1
140   846   -180.1316 -180.1316   2:59.4
142   852   -180.1316 -180.1316   3:00.7
Halting: No significant change in best function evaluation for 100 iterations.
[1.81193420e+00 1.00000004e-03] -180.13159583272244
Optimisation phase is finished.


In [20]:
routine_run(freq_samplying_range[2], sample_size_range[1])

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:144: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:160: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6
Iter. Eval. Best      Current   Time    
0     6     -179.1454 -179.1454   0:01.4
1     12    -179.1454 -179.445    0:02.6
2     18    -179.1454 -179.1608   0:03.8
3     24    -178.7032 -178.7032   0:05.3
20    126   -178.4498 -178.5032   0:21.3
40    246   -144.1848 -144.1848   0:42.2
60    366   -137.7527 -137.7648   1:06.0
80    486   -137.7134 -137.7134   1:26.4
100   606   -137.713  -137.713    1:47.0
120   726   -137.713  -137.713    2:04.3
140   846   -137.713  -137.713    2:24.1
144   864   -137.713  -137.713    2:28.0
Halting: No significant change in best function evaluation for 100 iterations.
[1.77698588e+00 1.00000000e-03] -137.71303296645792
Optimisation phase is finished.


In [21]:
routine_run(freq_samplying_range[3], sample_size_range[1])

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:144: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:160: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6
Iter. Eval. Best      Current   Time    
0     6     -82.60744 -82.60744   0:00.7
1     12    -81.95413 -81.95413   0:01.4
2     18    -81.61433 -81.61433   0:02.1
3     24    -81.61433 -81.65756   0:02.5
20    126   -81.44175 -81.49702   0:08.5
40    246   -81.40859 -81.40859   0:19.3
60    366   -79.44039 -79.44039   0:30.0
80    486   -58.13856 -58.13856   0:40.3
100   606   -57.65185 -57.67511   0:52.1
120   726   -57.64456 -57.64456   1:02.9
140   846   -57.64355 -57.64355   1:12.7
160   966   -57.64353 -57.64353   1:21.9
173   1038  -57.64353 -57.64353   1:27.1
Halting: No significant change in best function evaluation for 100 iterations.
[1.81962425e+00 1.00000013e-03] -57.64353304332689
Optimisation phase is finished.


In [22]:
routine_run(freq_samplying_range[0], sample_size_range[2])

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:144: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:160: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6
Iter. Eval. Best      Current   Time    
0     6     -493.0289 -493.0289   0:04.1
1     12    -493.0289 -493.0505   0:06.4
2     18    -493.0289 -493.2864   0:10.0
3     24    -493.0289 -493.2244   0:12.9
20    126   -492.0393 -492.1714   0:59.7
40    246   -407.342  -407.342    1:44.4
60    366   -393.2641 -393.3207   2:43.8
80    486   -393.2574 -393.2689   3:36.4
100   606   -393.2544 -393.2546   4:34.6
120   726   -393.2543 -393.2543   5:23.7
140   846   -393.2543 -393.2543   6:21.7
143   858   -393.2543 -393.2543   6:28.9
Halting: No significant change in best function evaluation for 100 iterations.
[1.84340152e+00 1.00000020e-03] -393.25432900483634
Optimisation phase is finished.


In [23]:
routine_run(freq_samplying_range[1], sample_size_range[2])

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:144: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:160: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6
Iter. Eval. Best      Current   Time    
0     6     -292.49   -292.49     0:02.6
1     12    -290.8695 -290.8695   0:04.8
2     18    -290.7188 -290.7188   0:06.9
3     24    -290.7159 -290.7159   0:09.1
20    126   -290.2822 -290.4178   0:33.6
40    246   -288.8594 -288.8594   1:03.4
60    366   -229.6345 -230.0463   1:40.7
80    486   -229.1892 -229.1892   2:16.7
100   606   -229.1799 -229.1802   2:48.9
120   726   -229.1793 -229.1793   3:23.9
140   846   -229.1793 -229.1793   3:49.9
160   966   -229.1793 -229.1793   4:23.5
171   1026  -229.1793 -229.1793   4:38.4
Halting: No significant change in best function evaluation for 100 iterations.
[1.83094466e+00 1.00000028e-03] -229.17925565084386
Optimisation phase is finished.


In [24]:
routine_run(freq_samplying_range[2], sample_size_range[2])

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:144: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:160: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6
Iter. Eval. Best      Current   Time    
0     6     -215.4376 -215.4376   0:01.5
1     12    -215.4376 -215.4897   0:03.2
2     18    -215.0164 -215.0164   0:04.7
3     24    -214.6882 -214.6882   0:06.2
20    126   -213.4658 -213.4658   0:24.1
40    246   -160.9406 -161.1953   0:48.9
60    366   -160.9406 -161.111    1:16.8
80    486   -160.9406 -161.1109   1:46.5
100   606   -160.9406 -161.1109   2:14.7
120   726   -160.9406 -161.1109   2:42.9
137   822   -160.9406 -161.1109   3:05.4
Halting: No significant change in best function evaluation for 100 iterations.
[1.6884112  0.00319498] -160.94062548414215
Optimisation phase is finished.


In [25]:
routine_run(freq_samplying_range[3], sample_size_range[2])

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:144: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:160: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6
Iter. Eval. Best      Current   Time    
0     6     -116.0213 -116.0213   0:01.0
1     12    -115.0861 -115.0861   0:02.1
2     18    -114.9507 -114.9507   0:02.9
3     24    -114.9507 -115.4526   0:03.2
20    126   -114.9054 -115.1356   0:14.8
40    246   -109.5879 -109.5879   0:30.0
60    366   -92.24577 -92.39897   0:45.2
80    486   -92.11861 -92.12786   0:59.0
100   606   -92.1155  -92.1155    1:16.5
120   726   -92.11441 -92.11441   1:27.5
140   846   -92.11441 -92.11441   1:41.7
152   912   -92.11441 -92.11441   1:50.7
Halting: No significant change in best function evaluation for 100 iterations.
[1.89572313e+00 1.00000046e-03] -92.11440657159162
Optimisation phase is finished.


### Repeat results with different start time

In [26]:
def routine_run_diff_start_time(freq_samplying, sample_size):
    sample_points = np.arange(5, 95, freq_samplying)

    # For each choice of sample size and frequency infer parameters: 
    vr_values_data, R0_found, shody_recov_freq, infec_freq_sample = sensitivity_analysis_run(sample_points, sample_size)

    mvr_inference = mmi.MVRVirReadInfer(algorithm, generation_times=generation_times)
    mvr_inference.read_viral_read_data(vr_values_data[0], parameters_vl)
    mvr_inference._create_posterior()

    parameters[4] = R0_found[1]
    parameters[6] = R0_found[0] / infect_period

    theta_found = []
    infec_found = []
    for _ in range(1000):
        output_found = algorithm.simulate_fixed_times(parameters, start_time, end_time)[0]

        theta_found.append(np.divide(output_found[:, 1], np.sum(output_found, axis=1)))
        infec_found.append(output_found[:, 1])

    theta_found = np.array(theta_found)
    infec_found = np.array(infec_found)

    theta_found_mean = np.mean(theta_found, axis=0)
    theta_found_upper = np.quantile(theta_found, 0.975, axis=0)
    theta_found_lower = np.quantile(theta_found, 0.025, axis=0)

    infec_found_mean = np.mean(infec_found, axis=0)
    infec_found_upper = np.quantile(infec_found, 0.975, axis=0)
    infec_found_lower = np.quantile(infec_found, 0.025, axis=0)

    output_found_det = mvr_inference.loglikelihood._run_sir_model(
        parameters, np.arange(max(mvr_inference.loglikelihood._vr_sampled_times))
    )
    theta_found_det = np.divide(output_found_det[:, 1], np.sum(output_found_det, axis=1))
    infec_found_det = output_found_det[:, 1]

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            y=(output_algorithm[0, :, 1] / np.mean(np.sum(output_algorithm, axis=2), axis=0)).tolist(),
            x=times,
            mode='lines',
            name='True freq',
            line_color='red'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=infec_freq_sample.tolist(),
            x=sample_points,
            mode='lines',
            name='Sample freq',
            line_color='blue'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=(1 - np.array(shody_recov_freq)).tolist(),
            x=sample_points,
            mode='lines',
            name='Sample freq (extreme I)',
            line_color='green'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=theta_found_det.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq - det',
            line_color='black',
            line_dash='dash'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=theta_found_mean.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq',
            line_color='black'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=times + times[::-1],
            y=theta_found_upper.tolist() + theta_found_lower.tolist()[::-1],
            fill='toself',
            fillcolor='black',
            line_color='black',
            opacity=0.3,
            mode='lines',
            showlegend=False,
            name='Inferred freq'
        )
    )

    # Add axis labels
    fig.update_layout(
        title='Sensitivity analysis: Frequency:{} days; Sample size:{}'.format(freq_samplying, sample_size),
        width=700, 
        height=400,
        plot_bgcolor='white',
        xaxis=dict(
            linecolor='black',
            title = 'Time (days)'
            ),
        yaxis=dict(
            linecolor='black',
            title = 'Individuals'),
        )

    fig.write_image('images/1Rerun_Optimal_sampling_Diff_Start_Viral_read_CredInt_Freq_{}_Sample_size_{}.pdf'.format(freq_samplying, sample_size))
    fig.show()

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            y=(output_algorithm[0, :, 1]).tolist(),
            x=times,
            mode='lines',
            name='True freq',
            line_color='red'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=infec_found_det.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq - det',
            line_color='black',
            line_dash='dash'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=infec_found_mean.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq',
            line_color='black'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=times + times[::-1],
            y=infec_found_upper.tolist() + infec_found_lower.tolist()[::-1],
            fill='toself',
            fillcolor='black',
            line_color='black',
            opacity=0.3,
            mode='lines',
            showlegend=False,
            name='Inferred freq'
        )
    )

    # Add axis labels
    fig.update_layout(
        title='Total Infections: Frequency:{} days; Sample size:{}'.format(freq_samplying, sample_size),
        width=700, 
        height=400,
        plot_bgcolor='white',
        xaxis=dict(
            linecolor='black',
            title = 'Time (days)'
            ),
        yaxis=dict(
            linecolor='black',
            title = 'Individuals'))

    fig.write_image('images/1Rerun_Total_Infec_Diff_Start_SIR_Viral_read__Freq_{}_Sample_size_{}.pdf'.format(freq_samplying, sample_size))
    fig.show()

In [27]:
routine_run_diff_start_time(freq_samplying_range[0], sample_size_range[2])

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_25178/3964525704.py:160: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6
Iter. Eval. Best      Current   Time    
0     6     -359.74   -359.74     0:04.3
1     12    -358.6122 -358.6122   0:07.1
2     18    -358.2507 -358.2507   0:08.8
3     24    -358.2507 -359.5433   0:10.5
20    126   -357.7076 -357.7076   0:54.4
40    246   -307.1761 -307.1761   1:40.4
60    366   -306.3574 -306.3961   2:33.0
80    486   -306.355  -306.3563   3:23.1
100   606   -306.3548 -306.3551   4:12.4
120   726   -306.3547 -306.3547   4:58.3
140   846   -306.3547 -306.3547   5:47.0
157   942   -306.3547 -306.3547   6:27.8
Halting: No significant change in best function evaluation for 100 iterations.
[2.05273916e+00 1.00000002e-03] -306.35467296063115
Optimisation phase is finished.


### Different infectious period and subsequently different viral read model dynamics

In [14]:
# Set initial reproduction number
R_0 = 3

# Set initial population state S - I - R
N_init = 400
# S_init = int(N_init / R_0)
S_init = 380
I_init = N_init - S_init
R_init = 0
initial_population = [S_init, I_init, R_init]

# Set birth rate
theta = 0.05

# Set death rates
mu = 0.002
nu = 0

# Set transition rates
infect_period = 40
beta =  R_0 / infect_period
gamma = 1 / infect_period

# Coalesce into paramater vector
parameters = initial_population
parameters.extend([theta, mu, nu, beta, gamma])

# Instantiate algorithm
algorithm = mm.Metaviromodel()

# Select start and end times
start_time = 1
end_time = 360

times = list(range(start_time, end_time+1))

# Select number of experiments
num_experiments = 1

output_algorithm = []

S_history_algorithm = []
I_history_algorithm = []
R_history_algorithm = []

I_times_history_algorithm = []
R_times_history_algorithm = []

for _ in range(num_experiments):
    output, S_history, I_history, R_history, I_times_history, R_times_history = algorithm.simulate_fixed_times(parameters, start_time, end_time)
    output_algorithm.append(output)

    S_history_algorithm.append(S_history)
    I_history_algorithm.append(I_history)
    R_history_algorithm.append(R_history)

    I_times_history_algorithm.append(I_times_history)
    R_times_history_algorithm.append(R_times_history)

output_algorithm = np.asarray(output_algorithm)

In [15]:
# Trace names - represent the type of individuals for the simulation
trace_name = ['{}'.format(s) for s in compartments]

# Names of panels
panels = ['{} only'.format(s) for s in compartments] + ['Total Population']

fig = go.Figure()
fig = make_subplots(rows=int(np.ceil(len(panels)/2)), cols=2, subplot_titles=tuple('{}'.format(p) for p in panels))

# Add traces to the separate counts panels
for s, spec in enumerate(compartments):
    fig.add_trace(
        go.Scatter(
            y=np.mean(output_algorithm[:, :, s], axis=0).tolist(),
            x=times,
            mode='lines',
            name=trace_name[s],
            line_color=colours[s]
        ),
        row= int(np.floor(s / 2)) + 1,
        col= s % 2 + 1
    )

fig.add_trace(
    go.Scatter(
        y=np.mean(np.sum(output_algorithm, axis=2), axis=0).tolist(),
        x=times,
        mode='lines',
        name='Total Population',
        line_color='black'
    ),
    row= 2,
    col= 2
)

# Add axis labels
fig.update_layout(
    title='Counts of compartments over time:<br>IC = {}, θ = {}, μ = {}, v = {}, β = {:.2f}, γ = {:.2f}'.format(parameters[0:3], parameters[3], parameters[4], parameters[5], parameters[6], parameters[7]),
    width=1100, 
    height=600,
    plot_bgcolor='white',
    xaxis=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis=dict(
        linecolor='black',
        title = 'Individuals'),
    xaxis2=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis2=dict(
        linecolor='black',
        title = 'Individuals'),
    xaxis3=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis3=dict(
        linecolor='black',
        title = 'Individuals'),
    xaxis4=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis4=dict(
        linecolor='black',
        title = 'Individuals')
    #legend=dict(
    #    orientation="h",
    #    yanchor="bottom",
    #    y=1.02,
    #    xanchor="right",
    #    x=1
    #)
    )

fig.write_image('images/Optimal-sampling-Diff_dynamics-gillespie.pdf')
fig.show()

In [23]:
t_eclipse = 6  # (0 days) Time from infection to initial viral growth
t_peak = 14  # (5 days ) Time from initial viral growth to peak viral load
t_switch = 20  # (9.38 days) Time from peak viral load to secondary waning phase
t_mod = 45  # (14 days) time from secondary waning phase until gumbel distribution reaches its min scale parameter
t_LOD = math.inf # ( inf days ) Time from infection until modal read counts value is equal to the limit of detection

sigma_obs = 0.25  # Initial scale parameter for the Gumbel distribution until a=teclipse+tpeak+tswitch
s_mod = 0.4  # 0.4 multiplicative factor applied to scale paramter for the Gumble distrbution - starting at t_eclipse + t_peak + t_switch + t_scle
v_zero = 2  # read counts value at time of infection
v_peak = 3880  # (20) Modal read counts value at peak viral load
v_switch = 480  # (33) Modal read counts value at a = teclipse + tpeak + tswitch
v_LOD = 2  # Limit of detection of read counts value

parameters_vl = [
    t_eclipse, t_peak, t_switch, t_mod, t_LOD,
    v_zero, v_peak, v_switch, v_LOD,
    s_mod, sigma_obs]

# Set read counts value for the suceptible and recovered individuals
VR_susc = 0

In [24]:
# Daily probability of recovered fully clearing the virus
p_addl = 0.2

R_history_clear_algorithm = []

# Go through each run experiment
for _ in range(num_experiments):
    R_history_clear = []

    # Go through each recorded day
    for t, time in enumerate(times):
        current_clear_status = []

        # If there are any recovered individual
        if len(R_times_history_algorithm[_][t]) > 0:
            # Go through each of them and
            for ind, ind_ID in enumerate(R_history_algorithm[_][t]):
                clear_status = 0

                # If they have previously cleared the virus they signal that
                if ind_ID in R_history_algorithm[_][t-1] and R_history_clear[-1][R_history_algorithm[_][t-1].index(ind_ID)] == 1:
                    clear_status = 1
                # if not, they could do it today, if their time since infection exceeds teclipse + tpeak + tswitch
                elif time > R_times_history_algorithm[_][t][ind] + t_eclipse + t_peak + t_switch:
                    clear_status = 1 - np.random.binomial(1, p = (1-p_addl)**(
                        time - R_times_history_algorithm[_][t][ind] - t_eclipse - t_peak - t_switch))

                current_clear_status.append(clear_status)

        R_history_clear.append(current_clear_status)                

    R_history_clear_algorithm.append(R_history_clear)

In [25]:
# Compute the generation times distribution, which also follows a
# right-skewed Gumbel distribution
generation_times = []

for _ in range(70):
    if _ < t_eclipse + t_peak + t_switch:
        generation_times.append(
            1-gumbel_r.cdf(
                np.log(v_LOD),
                algorithm._compute_mode_vr_model(
                    _, t_eclipse, t_peak, t_switch, t_LOD,
                    np.log(v_zero), np.log(v_peak), np.log(v_switch), np.log(v_LOD)),
                algorithm._compute_sigma_vr_model(
                    _, t_eclipse, t_peak, t_switch, t_mod,
                    s_mod, sigma_obs)
            ))
        
    else:
        generation_times.append(
            (1-gumbel_r.cdf(
                np.log(v_LOD),
                algorithm._compute_mode_vr_model(
                    _, t_eclipse, t_peak, t_switch, t_LOD,
                    np.log(v_zero), np.log(v_peak), np.log(v_switch), np.log(v_LOD)),
                algorithm._compute_sigma_vr_model(
                    _, t_eclipse, t_peak, t_switch, t_mod,
                    s_mod, sigma_obs)
            )) * (1-p_addl)**(_ - t_eclipse - t_peak - t_switch))

In [26]:
# Transform birth rate and death rate of infected into function format for inference method
parameters[3] = lambda _: theta
parameters[5] = lambda _: nu

In [27]:
def routine_run_different_dynamics(freq_samplying, sample_size):
    sample_points = np.arange(20, 210, freq_samplying)

    # For each choice of sample size and frequency infer parameters: 
    vr_values_data, R0_found, shody_recov_freq, infec_freq_sample = sensitivity_analysis_run(sample_points, sample_size)

    mvr_inference = mmi.MVRVirReadInfer(algorithm, generation_times=generation_times)
    mvr_inference.read_viral_read_data(vr_values_data[0], parameters_vl)
    mvr_inference._create_posterior()

    parameters[4] = R0_found[1]
    parameters[6] = R0_found[0] / infect_period

    theta_found = []
    infec_found = []
    for _ in range(1000):
        output_found = algorithm.simulate_fixed_times(parameters, start_time, end_time)[0]

        theta_found.append(np.divide(output_found[:, 1], np.sum(output_found, axis=1)))
        infec_found.append(output_found[:, 1])

    theta_found = np.array(theta_found)
    infec_found = np.array(infec_found)

    theta_found_mean = np.mean(theta_found, axis=0)
    theta_found_upper = np.quantile(theta_found, 0.975, axis=0)
    theta_found_lower = np.quantile(theta_found, 0.025, axis=0)

    infec_found_mean = np.mean(infec_found, axis=0)
    infec_found_upper = np.quantile(infec_found, 0.975, axis=0)
    infec_found_lower = np.quantile(infec_found, 0.025, axis=0)

    output_found_det = mvr_inference.loglikelihood._run_sir_model(
        parameters, np.arange(max(mvr_inference.loglikelihood._vr_sampled_times))
    )
    theta_found_det = np.divide(output_found_det[:, 1], np.sum(output_found_det, axis=1))
    infec_found_det = output_found_det[:, 1]

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            y=(output_algorithm[0, :, 1] / np.mean(np.sum(output_algorithm, axis=2), axis=0)).tolist(),
            x=times,
            mode='lines',
            name='True freq',
            line_color='red'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=infec_freq_sample.tolist(),
            x=sample_points,
            mode='lines',
            name='Sample freq',
            line_color='blue'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=(1 - np.array(shody_recov_freq)).tolist(),
            x=sample_points,
            mode='lines',
            name='Sample freq (extreme I)',
            line_color='green'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=theta_found_det.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq - det',
            line_color='black',
            line_dash='dash'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=theta_found_mean.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq',
            line_color='black'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=times + times[::-1],
            y=theta_found_upper.tolist() + theta_found_lower.tolist()[::-1],
            fill='toself',
            fillcolor='black',
            line_color='black',
            opacity=0.3,
            mode='lines',
            showlegend=False,
            name='Inferred freq'
        )
    )

    # Add axis labels
    fig.update_layout(
        title='Sensitivity analysis: Frequency:{} days; Sample size:{}'.format(freq_samplying, sample_size),
        width=700, 
        height=400,
        plot_bgcolor='white',
        xaxis=dict(
            linecolor='black',
            title = 'Time (days)'
            ),
        yaxis=dict(
            linecolor='black',
            title = 'Individuals'),
        )

    fig.write_image('images/1Rerun_Optimal_sampling_Diff_Dynamics_Viral_read_CredInt_Freq_{}_Sample_size_{}.pdf'.format(freq_samplying, sample_size))
    fig.show()

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            y=(output_algorithm[0, :, 1]).tolist(),
            x=times,
            mode='lines',
            name='True freq',
            line_color='red'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=infec_found_det.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq - det',
            line_color='black',
            line_dash='dash'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=infec_found_mean.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq',
            line_color='black'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=times + times[::-1],
            y=infec_found_upper.tolist() + infec_found_lower.tolist()[::-1],
            fill='toself',
            fillcolor='black',
            line_color='black',
            opacity=0.3,
            mode='lines',
            showlegend=False,
            name='Inferred freq'
        )
    )

    # Add axis labels
    fig.update_layout(
        title='Total Infections: Frequency:{} days; Sample size:{}'.format(freq_samplying, sample_size),
        width=700, 
        height=400,
        plot_bgcolor='white',
        xaxis=dict(
            linecolor='black',
            title = 'Time (days)'
            ),
        yaxis=dict(
            linecolor='black',
            title = 'Individuals'))

    fig.write_image('images/1Rerun_Total_Infec_Diff_Dynamics_SIR_Viral_read__Freq_{}_Sample_size_{}.pdf'.format(freq_samplying, sample_size))
    fig.show()

In [28]:
routine_run_different_dynamics(freq_samplying_range[0], sample_size_range[2])

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_27385/3964525704.py:144: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_27385/3964525704.py:160: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6
Iter. Eval. Best      Current   Time    
0     6     -757.0806 -757.0806   0:05.7
1     12    -751.8054 -751.8054   0:11.4
2     18    -747.1185 -747.1185   0:17.5
3     24    -747.1185 -747.3516   0:21.6
20    126   -745.2114 -745.2218   1:28.6
40    246   -745.2083 -745.2149   2:48.8
60    366   -745.1949 -745.1949   4:02.1
80    486   -730.9736 -730.9736   5:27.1
100   606   -657.0773 -657.252    6:59.5
120   726   -656.8914 -656.8957   8:24.1
140   846   -656.8875 -656.8875   9:42.4
160   966   -656.8875 -656.8875  11:02.1
180   1086  -656.8875 -656.8875  12:24.0
199   1194  -656.8875 -656.8875  13:34.9
Halting: No significant change in best function evaluation for 100 iterations.
[2.10440181e+00 1.00000000e-03] -656.8874865010021
Optimisation phase is finished.


In [29]:
def routine_run_different_dynamics_diff_start(freq_samplying, sample_size):
    sample_points = np.arange(5, 195, freq_samplying)

    # For each choice of sample size and frequency infer parameters: 
    vr_values_data, R0_found, shody_recov_freq, infec_freq_sample = sensitivity_analysis_run(sample_points, sample_size)

    mvr_inference = mmi.MVRVirReadInfer(algorithm, generation_times=generation_times)
    mvr_inference.read_viral_read_data(vr_values_data[0], parameters_vl)
    mvr_inference._create_posterior()

    parameters[4] = R0_found[1]
    parameters[6] = R0_found[0] / infect_period

    theta_found = []
    infec_found = []
    for _ in range(1000):
        output_found = algorithm.simulate_fixed_times(parameters, start_time, end_time)[0]

        theta_found.append(np.divide(output_found[:, 1], np.sum(output_found, axis=1)))
        infec_found.append(output_found[:, 1])

    theta_found = np.array(theta_found)
    infec_found = np.array(infec_found)

    theta_found_mean = np.mean(theta_found, axis=0)
    theta_found_upper = np.quantile(theta_found, 0.975, axis=0)
    theta_found_lower = np.quantile(theta_found, 0.025, axis=0)

    infec_found_mean = np.mean(infec_found, axis=0)
    infec_found_upper = np.quantile(infec_found, 0.975, axis=0)
    infec_found_lower = np.quantile(infec_found, 0.025, axis=0)

    output_found_det = mvr_inference.loglikelihood._run_sir_model(
        parameters, np.arange(max(mvr_inference.loglikelihood._vr_sampled_times))
    )
    theta_found_det = np.divide(output_found_det[:, 1], np.sum(output_found_det, axis=1))
    infec_found_det = output_found_det[:, 1]

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            y=(output_algorithm[0, :, 1] / np.mean(np.sum(output_algorithm, axis=2), axis=0)).tolist(),
            x=times,
            mode='lines',
            name='True freq',
            line_color='red'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=infec_freq_sample.tolist(),
            x=sample_points,
            mode='lines',
            name='Sample freq',
            line_color='blue'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=(1 - np.array(shody_recov_freq)).tolist(),
            x=sample_points,
            mode='lines',
            name='Sample freq (extreme I)',
            line_color='green'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=theta_found_det.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq - det',
            line_color='black',
            line_dash='dash'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=theta_found_mean.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq',
            line_color='black'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=times + times[::-1],
            y=theta_found_upper.tolist() + theta_found_lower.tolist()[::-1],
            fill='toself',
            fillcolor='black',
            line_color='black',
            opacity=0.3,
            mode='lines',
            showlegend=False,
            name='Inferred freq'
        )
    )

    # Add axis labels
    fig.update_layout(
        title='Sensitivity analysis: Frequency:{} days; Sample size:{}'.format(freq_samplying, sample_size),
        width=700, 
        height=400,
        plot_bgcolor='white',
        xaxis=dict(
            linecolor='black',
            title = 'Time (days)'
            ),
        yaxis=dict(
            linecolor='black',
            title = 'Individuals'),
        )

    fig.write_image('images/1Rerun_Optimal_sampling_Diff_Dynamics_Diff_start_Viral_read_CredInt_Freq_{}_Sample_size_{}.pdf'.format(freq_samplying, sample_size))
    fig.show()

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            y=(output_algorithm[0, :, 1]).tolist(),
            x=times,
            mode='lines',
            name='True freq',
            line_color='red'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=infec_found_det.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq - det',
            line_color='black',
            line_dash='dash'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=infec_found_mean.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq',
            line_color='black'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=times + times[::-1],
            y=infec_found_upper.tolist() + infec_found_lower.tolist()[::-1],
            fill='toself',
            fillcolor='black',
            line_color='black',
            opacity=0.3,
            mode='lines',
            showlegend=False,
            name='Inferred freq'
        )
    )

    # Add axis labels
    fig.update_layout(
        title='Total Infections: Frequency:{} days; Sample size:{}'.format(freq_samplying, sample_size),
        width=700, 
        height=400,
        plot_bgcolor='white',
        xaxis=dict(
            linecolor='black',
            title = 'Time (days)'
            ),
        yaxis=dict(
            linecolor='black',
            title = 'Individuals'))

    fig.write_image('images/1Rerun_Total_Infec_Diff_Dynamics_Diff_start_SIR_Viral_read__Freq_{}_Sample_size_{}.pdf'.format(freq_samplying, sample_size))
    fig.show()

In [30]:
routine_run_different_dynamics_diff_start(freq_samplying_range[0], sample_size_range[2])

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_27385/3964525704.py:160: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Maximising LogPDF
Using Covariance Matrix Adaptation Evolution Strategy (CMA-ES)
Running in sequential mode.
Population size: 6
Iter. Eval. Best      Current   Time    
0     6     -692.0342 -692.0342   0:05.5
1     12    -692.0342 -692.2979   0:09.7
2     18    -687.8799 -687.8799   0:15.9
3     24    -687.6749 -687.6749   0:22.1
20    126   -685.8117 -685.9315   1:26.4
40    246   -685.6134 -685.6134   2:47.6
60    366   -636.9095 -638.7951   4:15.7
80    486   -633.9729 -633.9729   5:38.0
100   606   -633.9241 -633.9264   6:59.2
120   726   -633.9231 -633.9232   8:26.3
140   846   -633.9224 -633.9224   9:50.0
160   966   -633.9224 -633.9224  11:12.8
168   1008  -633.9224 -633.9224  11:43.4
Halting: No significant change in best function evaluation for 100 iterations.
[2.25488570e+00 1.00000031e-03] -633.9224336901785
Optimisation phase is finished.
